# Design Spaces 

## Setup

### Import Packages and Directories

In [1]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path

from flowtangent.models.kernels import RBFKernel, Matern52Kernel
from flowtangent.models.surrogates import SVGPSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough, ScalingManifold
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

import jax.numpy as jnp

In [2]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name: str):
    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(batch: dict):
    names = batch["foil_name"]
    
    # 1. Strip Xarray metadata if it slipped through
    if hasattr(names, "values"):
        names = names.values
        
    # 2. Force it into a flat, pure NumPy array
    names = np.asarray(names).flatten()
    
    # 3. Now iteration yields pure strings (or numpy.str_), which str() handles perfectly
    surfaces = [surf_lookup(str(name)) for name in names]
    
    batch["y_af"] = jnp.stack(surfaces)
    return batch

### Import Dataset

In [3]:
data_dir = Path("/home/jordan/dev/datasets/mUIUC_5D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="eden-uiuc/mUIUC-5D", repo_type="dataset", filename="mUIUC_5D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting mUIUC 5D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

Stored dataset detected.
Dataset ready at: /home/jordan/dev/datasets/mUIUC_5D.zarr


In [5]:
XFD_arr = ft.utils.data.XArrayDataset(data_dir)

In [6]:
XFD_raw = ft.utils.data.XArrayDataset(data_dir).to_tabular(["run", "alpha"])
XFD = XFD_raw.dropna(["alpha"])
XFD = XFD.transform_variable(name2surf)
XFD._proxy._ds.load()

Dropping NaNs from dataset:   0%|          | 0/104 [00:00<?, ?it/s]

<xarray.Dataset> Size: 654MB
Dimensions:     (run: 358400, bend_var: 24, cond_var: 4, alpha: 42, polar_var: 9)
Coordinates:
  * bend_var    (bend_var) <U18 2kB 'Area' 'Slen' ... 'skin_Y_Ixx/t(Y-Yc)' 'J/t'
  * cond_var    (cond_var) <U10 160B 'flap_angle' 'flap_x/c' 'Re' 'Mach'
  * polar_var   (polar_var) <U7 252B 'alpha' 'CL' 'CD' ... 'Top_Itr' 'Bot_Itr'
Dimensions without coordinates: run, alpha
Data variables:
    bend_data   (run, bend_var) float32 34MB 0.0463 2.051 0.3786 ... 0.04929 nan
    conditions  (run, cond_var) float32 6MB -1.81 0.7673 ... 1.978e+05 0.5435
    foil_name   (run) <U50 72MB '2032c' '2032c' '2032c' ... 'ys930' 'ys930'
    polar_data  (run, alpha, polar_var) float32 542MB 0.0 0.5962 ... 5.937 86.99

In [7]:
unique_names = [str(n) for n in set(np.asarray(XFD_raw.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Set up Parameters

In [8]:
# Inputs
y_af = Parameter(name="y_af", is_transform=True, transform_sources=("foil_name",))
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)
CD = Parameter(name="CD", is_output=True)
CM = Parameter(name="CM", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha, CL, CD, CM)

### Set up Manifold

In [9]:
geofold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
geofold = geofold.fit(X_SURF)

machfold    = ScalingManifold(method="softplus")
refold      = ScalingManifold(method="logarithmic")
hingefold   = ScalingManifold(method="algebraic_bounded", bounds_min=0.6, bounds_max=0.85)
deltafold   = ScalingManifold(method="linear", scale_factors=10)
alphafold   = ScalingManifold(method="linear", scale_factors=15)

mfold = geofold + machfold + refold + hingefold + deltafold + alphafold

### Setup Design Space

In [10]:
batch_size = 8192
beta = 0.5 * batch_size / len(XFD)

mu_scl =  (XFD.CL.mean().item(), XFD.CD.mean().item(), XFD.CM.mean().item())
sig_scl = (XFD.CL.std().item(), XFD.CD.std().item(), XFD.CM.std().item())

krnl = Matern52Kernel(input_dim=21)
surr = SVGPSurrogate(
    jax.random.key(42),
    input_dim=21,
    output_dim=3,
    kernel=krnl,
    num_inducing=256,
    complexity_weight=beta,
    jitter=1e-5,
    # scale_mean=mu_scl,
    # scale_std=sig_scl
)

airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
)

In [ ]:
airspace = airspace.fit_models(batch_size=batch_size, surrogate_epochs=5)

Training Neural Model (notebook mode):   0%|          | 0/5 [00:00<?, ?it/s]

FigureWidget({
    'data': [{'line': {'color': '#2563eb'},
              'mode': 'lines',
              'name': 'Loss',
              'type': 'scatter',
              'uid': '481fd0f4-b570-4303-9aee-406e3d2f4f9b',
              'y': []}],
    'layout': {'template': '...', 'title': {'text': 'Training Convergence'}, 'yaxis': {'type': 'log'}}
})

Epoch 1 Batches:   0%|          | 0/1585 [00:00<?, ?it/s]

Epoch 2 Batches:   0%|          | 0/1585 [00:00<?, ?it/s]

In [27]:
airspace.evaluate(airspace.project(XFD[0])).variances

Array([2.29867672e-02, 1.45155012e-05, 1.43599425e-05], dtype=float64)

In [13]:
evals = [airspace.evaluate(airspace.project(XFD[i])) for i in range(5)]

In [14]:
metrics = ["CL", "CD", "CM"]

for i, e in enumerate(evals):
    print(f"CL: Pred - {e.means[0]:.3f} ({e.variances[0]:.3f}); Target: [{XFD[i]["CL"]:.3f}]({(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"] * 100 :.2f}%, {(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"]/e.variances[0]:.2f})")
    print(f"CD: Pred - {e.means[1]:.3f} ({e.variances[1]:.3f}); Target: [{XFD[i]["CD"]:.3f}]({(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"] * 100 :.2f}%, {(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"]/e.variances[1]:.2f})")
    print(f"CM: Pred - {e.means[2]:.3f} ({e.variances[2]:.3f}); Target: [{XFD[i]["CM"]:.3f}]({(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"] * 100 :.2f}%, {(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"]/e.variances[2]:.2f})\n")

CL: Pred - 0.298 (0.023); Target: [0.596](-50.04%, -21.77)
CD: Pred - 0.001 (0.000); Target: [0.036](-97.63%, -67261.58)
CM: Pred - -0.006 (0.000); Target: [-0.131](-95.44%, -66459.59)

CL: Pred - 0.360 (0.023); Target: [0.729](-50.56%, -22.05)
CD: Pred - 0.001 (0.000); Target: [0.033](-98.13%, -71433.33)
CM: Pred - -0.006 (0.000); Target: [-0.130](-95.47%, -70306.52)

CL: Pred - 0.419 (0.023); Target: [0.841](-50.11%, -21.93)
CD: Pred - 0.000 (0.000); Target: [0.030](-98.45%, -77766.27)
CM: Pred - -0.006 (0.000); Target: [-0.125](-95.34%, -76286.90)

CL: Pred - 0.475 (0.023); Target: [0.944](-49.66%, -21.83)
CD: Pred - 0.000 (0.000); Target: [0.034](-98.76%, -86911.07)
CM: Pred - -0.006 (0.000); Target: [-0.123](-95.31%, -85117.18)

CL: Pred - 0.527 (0.023); Target: [1.021](-48.40%, -21.37)
CD: Pred - 0.000 (0.000); Target: [0.042](-98.84%, -99121.72)
CM: Pred - -0.006 (0.000); Target: [-0.121](-95.34%, -97270.12)



In [24]:
for i, e in enumerate(evals):
    print(f"CL: Pred - {e.means[0]:.3f} ({e.variances[0]:.3f}); Target: [{XFD[i]["CL"]:.3f}]({(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"] * 100 :.2f}%, {(e.means[0]- XFD[i]["CL"])/XFD[i]["CL"]/e.variances[0]:.2f})")
    print(f"CD: Pred - {e.means[1]:.3f} ({e.variances[1]:.3f}); Target: [{XFD[i]["CD"]:.3f}]({(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"] * 100 :.2f}%, {(e.means[1]- XFD[i]["CD"])/XFD[i]["CD"]/e.variances[1]:.2f})")
    print(f"CM: Pred - {e.means[2]:.3f} ({e.variances[2]:.3f}); Target: [{XFD[i]["CM"]:.3f}]({(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"] * 100 :.2f}%, {(e.means[2]- XFD[i]["CM"])/XFD[i]["CM"]/e.variances[2]:.2f})\n")

CL: Pred - 0.382 (0.032); Target: [0.596](-35.99%, -11.32)
CD: Pred - 0.029 (0.001); Target: [0.036](-19.84%, -136.79)
CM: Pred - -0.112 (0.001); Target: [-0.131](-14.80%, -107.19)

CL: Pred - 0.492 (0.032); Target: [0.729](-32.49%, -10.25)
CD: Pred - 0.030 (0.001); Target: [0.033](-9.29%, -68.90)
CM: Pred - -0.113 (0.001); Target: [-0.130](-13.35%, -104.40)

CL: Pred - 0.600 (0.032); Target: [0.841](-28.65%, -9.04)
CD: Pred - 0.031 (0.001); Target: [0.030](2.52%, 18.86)
CM: Pred - -0.113 (0.001); Target: [-0.125](-9.86%, -77.93)

CL: Pred - 0.703 (0.032); Target: [0.944](-25.57%, -8.06)
CD: Pred - 0.032 (0.001); Target: [0.034](-4.72%, -34.72)
CM: Pred - -0.111 (0.001); Target: [-0.123](-9.21%, -71.52)

CL: Pred - 0.798 (0.032); Target: [1.021](-21.88%, -6.89)
CD: Pred - 0.035 (0.001); Target: [0.042](-16.20%, -117.03)
CM: Pred - -0.109 (0.001); Target: [-0.121](-10.60%, -80.63)



In [16]:
dae41_1 = int(np.where(XFD.foil_name == "dae41")[0][0])

In [20]:
jnp.linalg.norm(airspace.project(XFD[dae41_1])[:16])

Array(0.03270292, dtype=float64)